# Preprocessing Lengkap — Cleaning Data TravelFit (443 Destinasi)

**Sumber:** 437 Kaggle (`data/raw/tourism_with_id.csv` + `tourism_rating.csv`) + 6 tambahan Jawa (`data/review/gabungan_jawa/destinasi_jawa_review.csv` + `google_maps_rating_review.json`).
**Kebijakan retensi:** TIDAK ada destinasi yang dihapus. Wajib tepat **443 ID unik 1-443** dari awal sampai ekspor.
**Output:** `data/processed/destinations_clean_java443.csv`, `data/processed/destinations_kmeans_features_java443.csv`, `reports/preprocessing/preprocessing_java443_summary.json`, `preprocessing_java443_manifest.json`.

## Peta Algoritma Cleaning (14 langkah)
1. Setup path & helper (hash, fingerprint, snake_case, keyword-boundary)
2. Load 4 sumber wajib (gagal cepat bila hilang/parsial)
3. Audit kualitas awal / EDA (missing, duplikat, distribusi, outlier, kolom sampah)
4. Cleaning kolom (snake_case, buang Unnamed, gabung 437+6 + provenance)
5. Cleaning teks (rapikan whitespace, kosong jadi NA audit)
6. Cleaning numerik (coerce + flag invalid per baris)
7. Treatment missing value (hanya median Tahura ID438 untuk kolom model)
8. Dedup & validasi ID (destinasi 1-443 unik; interaksi user x place keep-last + agregat)
9. Standardisasi wilayah & kategori (city ke province, category mapping)
10. Validasi geografis (range lat/long + flag Marina ID9, koordinat tidak dipindah)
11. Outlier harga (cap P99 hanya untuk fitur; harga master utuh)
12. Fitur fasilitas C4 + activity tags (keyword + koreksi konteks)
13. Fitur model K-Means (Z-score + one-hot)
14. Validasi akhir 443 + ekspor CSV/JSON + fingerprint manifest

> Jalankan cell berurutan dari atas ke bawah. Kompatibel lokal & Colab. Hanya butuh pandas + numpy.


In [ ]:
from pathlib import Path
import sys, re, json, hashlib
import numpy as np
import pandas as pd

if 'PROJECT_ROOT' not in globals():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_ROOT = Path('/content/drive/MyDrive/TravelFit')
    except ImportError:
        PROJECT_ROOT = Path.cwd()
        if PROJECT_ROOT.name == 'notebooks':
            PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = Path(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)

PIPELINE_VERSION = 'java443-v1'
SOURCES = {
    'kaggle_base': 'data/raw/tourism_with_id.csv',
    'tambahan': 'data/review/gabungan_jawa/destinasi_jawa_review.csv',
    'rating_receipt': 'data/review/gabungan_jawa/google_maps_rating_review.json',
    'interaksi': 'data/raw/tourism_rating.csv',
}
OUTPUTS = {
    'destinations': 'data/processed/destinations_clean_java443.csv',
    'features': 'data/processed/destinations_kmeans_features_java443.csv',
    'summary': 'reports/preprocessing/preprocessing_java443_summary.json',
    'manifest': 'reports/preprocessing/preprocessing_java443_manifest.json',
}
BASE_URL = 'https://raw.githubusercontent.com/akhiyarwaladi/indonesia_tourism/main/tourism_with_id.csv'

def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def fingerprint(payload: dict) -> str:
    s = json.dumps(payload, sort_keys=True, separators=(',', ':'), ensure_ascii=True, allow_nan=False)
    return hashlib.sha256(s.encode()).hexdigest()

def snake_case(v):
    return re.sub(r'[^0-9a-zA-Z]+', '_', str(v).strip()).strip('_').lower()

def normalize_text(v):
    return re.sub(r'\s+', ' ', str(v).casefold()).strip() if pd.notna(v) else ''

def contains_keyword(text: str, keyword: str) -> bool:
    esc = re.escape(keyword.casefold()).replace(r'\ ', r'\s+')
    return re.search(r'(?<!\w)' + esc + r'(?!\w)', text) is not None

def check_ids(frame, label='frame'):
    ok = len(frame) == 443 and frame['place_id'].is_unique and set(frame['place_id']) == set(range(1, 444))
    assert ok, label + ': wajib tepat 443 ID unik 1-443'

pd.set_option('display.max_columns', 30)
print('pandas', pd.__version__, '| numpy', np.__version__)


## 1. Load 4 sumber wajib
Algoritma: cek keberadaan file, baca CSV/JSON, validasi jumlah baris dan rentang ID (437 = ID 1-437, tambahan = ID 438-443). Gagal cepat bila parsial, jangan lanjut dengan dataset tidak lengkap.


In [ ]:
for key, rel in SOURCES.items():
    p = PROJECT_ROOT / rel
    assert p.is_file(), 'Sumber wajib hilang: ' + rel
    print('[OK]', key, rel, str(round(p.stat().st_size / 1024, 1)) + ' KB')

base = pd.read_csv(PROJECT_ROOT / SOURCES['kaggle_base'])
tambahan = pd.read_csv(PROJECT_ROOT / SOURCES['tambahan'])
interaksi_raw = pd.read_csv(PROJECT_ROOT / SOURCES['interaksi'])
receipt = json.loads((PROJECT_ROOT / SOURCES['rating_receipt']).read_text(encoding='utf-8'))

assert len(base) == 437 and set(base['Place_Id']) == set(range(1, 438))
assert len(tambahan) == 6 and set(tambahan['Place_Id']) == set(range(438, 444))
print('base:', base.shape, '| tambahan:', tambahan.shape, '| interaksi:', interaksi_raw.shape)
print('receipt access_date:', receipt.get('access_date'), '| records:', len(receipt.get('records', [])))
display(base.head(3))
display(tambahan[['Place_Id', 'Place_Name', 'Category', 'City', 'Price', 'Rating', 'Time_Minutes']])


## 2. Audit kualitas awal (EDA) — sebelum cleaning apa pun
Algoritma audit: bentuk dan dtype, persen missing per kolom, baris duplikat penuh, describe numerik, value_counts kategori/kota, outlier kasar (harga > 500rb, rating di luar 1-5, lat/long di luar range), kolom sampah Unnamed. Hasil audit menjadi acuan seluruh aturan cleaning.


In [ ]:
print('===== A. Bentuk & tipe =====')
print(base.dtypes.to_string())
print(interaksi_raw.dtypes.to_string())

print('===== B. Missing (%) =====')
miss = pd.concat([base.isna().mean().mul(100).round(2).rename('kaggle_pct'),
                  tambahan.isna().mean().mul(100).round(2).rename('tambahan_pct')], axis=1)
display(miss)
print('interaksi missing:')
print(interaksi_raw.isna().sum().to_string())

print('===== C. Duplikat =====')
print('base duplikat penuh:', base.duplicated().sum())
print('tambahan duplikat penuh:', tambahan.duplicated().sum())
print('interaksi duplikat user x place:', interaksi_raw.duplicated(subset=['User_Id', 'Place_Id']).sum())

print('===== D. Distribusi numerik =====')
display(base[['Price', 'Rating', 'Time_Minutes', 'Lat', 'Long']].describe().round(3))
display(interaksi_raw['Place_Ratings'].describe().round(3))

print('===== E. Kategori & kota =====')
print(base['Category'].value_counts().to_string())
print(base['City'].value_counts().to_string())

print('===== F. Outlier / invalid kasar =====')
print('harga == 0 (gratis, VALID):', (base['Price'] == 0).sum())
print('harga < 0:', (base['Price'] < 0).sum())
print('harga > 500rb:', (base['Price'] > 500000).sum())
print('rating di luar 1-5:', ((base['Rating'] < 1) | (base['Rating'] > 5)).sum())
print('lat di luar +-90:', (~base['Lat'].between(-90, 90)).sum())
print('long di luar +-180:', (~base['Long'].between(-180, 180)).sum())
print('Time kosong base:', base['Time_Minutes'].isna().sum())
print('kolom sampah:', [c for c in base.columns if str(c).startswith('Unnamed')])


## 3. Cleaning kolom: snake_case + buang Unnamed + gabung 437+6
Algoritma: beri metadata provenance sebelum concat (source_dataset, source_url, rating_status), terapkan rating Google Maps yang lolos verifikasi matched_name_location, gabung, normalisasi header ke snake_case, buang kolom unnamed, sort place_id.


In [ ]:
base['source_dataset'] = 'Indonesia Tourism Destination'
base['source_url'] = BASE_URL
base['rating_status'] = 'dataset_original'
base['rating_source_url'] = BASE_URL
base['rating_access_date'] = ''

tambahan['source_dataset'] = 'Tambahan Jawa: tarif peraturan daerah dan lokasi OSM'
tambahan['source_url'] = 'data/review/gabungan_jawa/Dokumentasi.md'
tambahan['rating_status'] = 'missing'
tambahan['rating_source_url'] = ''
tambahan['rating_access_date'] = ''

n_applied = 0
for rec in receipt['records']:
    m = tambahan['Place_Id'].eq(rec['place_id']) & tambahan['Place_Name'].eq(rec['dataset_name'])
    if rec.get('integration_allowed') and rec.get('match_status') == 'matched_name_location':
        tambahan.loc[m, 'Rating'] = rec['rating_observed']
        tambahan.loc[m, 'rating_status'] = 'google_maps_matched'
        tambahan.loc[m, 'rating_source_url'] = rec['maps_url']
        tambahan.loc[m, 'rating_access_date'] = receipt.get('access_date', '')
        n_applied += int(m.sum())
    else:
        tambahan.loc[m, 'rating_status'] = 'pending_identity'
print('rating Google Maps diterapkan:', n_applied, '(Tahura tetap missing by design)')

raw = pd.concat([base, tambahan], ignore_index=True)
raw.columns = [snake_case(c) for c in raw.columns]
drop_cols = [c for c in raw.columns if c.startswith('unnamed')]
raw = raw.drop(columns=drop_cols)
print('kolom dibuang:', drop_cols, '| shape:', raw.shape)
raw = raw.sort_values('place_id').reset_index(drop=True)
check_ids(raw, 'raw gabungan')
print('ID gabungan OK: 1-443 unik')
display(raw[['place_id', 'place_name', 'category', 'city', 'price', 'rating', 'rating_status']].tail(8))


## 4. Cleaning teks: place_name, description, category, city
Algoritma: cast string, collapse whitespace (s+ jadi satu spasi), strip. String kosong dianggap missing untuk audit. Kapitalisasi asli dipertahankan; pencocokan keyword memakai normalize_text terpisah.


In [ ]:
destinations = raw.copy()
for col in ['place_name', 'description', 'category', 'city']:
    destinations[col] = destinations[col].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
    n_empty = int((destinations[col].isna() | (destinations[col] == '')).sum())
    print(col, ': kosong/NA setelah rapi =', n_empty)
lens = destinations['description'].fillna('').str.len()
print(lens.describe().round(1).to_string())
print('deskripsi kosong (ditandai, TIDAK drop):', int((destinations['description'].isna() | (destinations['description'] == '')).sum()))


## 5. Cleaning numerik + flag kualitas per baris
Algoritma: to_numeric(errors=coerce) untuk place_id, price, rating, time_minutes, lat, long. Bangun kolom data_quality_issues (pipe-separated): id_missing, id_duplicate, name_missing, price_invalid, rating_missing, rating_out_of_range, latitude_invalid, longitude_invalid, coordinate_location_conflict. Baris bermasalah DITANDAI, bukan dibuang.


In [ ]:
for col in ['place_id', 'price', 'rating', 'time_minutes', 'lat', 'long']:
    destinations[col] = pd.to_numeric(destinations[col], errors='coerce')

destinations['coordinate_review_required'] = (destinations['place_id'].eq(9)
    & destinations['place_name'].eq('Pelabuhan Marina')
    & np.isclose(destinations['lat'], 1.07888)
    & np.isclose(destinations['long'], 103.931398))
destinations['coordinate_review_note'] = ''
destinations.loc[destinations['coordinate_review_required'], 'coordinate_review_note'] = (
    'Label Jakarta bertentangan dengan koordinat sumber di luar Jawa; baris dipertahankan sesuai keputusan pengguna.')

QUALITY_CHECKS = {
    'id_missing': destinations['place_id'].isna(),
    'id_duplicate': destinations['place_id'].duplicated(keep=False),
    'name_missing': destinations['place_name'].isna() | destinations['place_name'].eq(''),
    'price_invalid': ~destinations['price'].ge(0),
    'rating_missing': destinations['rating'].isna(),
    'rating_out_of_range': destinations['rating'].notna() & ~destinations['rating'].between(1, 5),
    'latitude_invalid': ~destinations['lat'].between(-90, 90),
    'longitude_invalid': ~destinations['long'].between(-180, 180),
    'coordinate_location_conflict': destinations['coordinate_review_required'],
}
destinations['data_quality_issues'] = ''
for issue, mask in QUALITY_CHECKS.items():
    destinations.loc[mask.fillna(False), 'data_quality_issues'] += issue + '|'
destinations['data_quality_issues'] = destinations['data_quality_issues'].str.rstrip('|')
print('Baris ber-flag kualitas:', int((destinations['data_quality_issues'] != '').sum()))
display(destinations.loc[destinations['data_quality_issues'] != '',
    ['place_id', 'place_name', 'price', 'rating', 'lat', 'long', 'data_quality_issues']])


## 6. Treatment missing value
Matriks kebijakan: rating sumber dipertahankan NaN (Tahura ID438). Kolom model c2_rating_for_model diisi median 442 teramati HANYA untuk Tahura. Time_minutes boleh kosong. Price/lat/long/category wajib ada. Description boleh kosong + flag. Rating hilang selain Tahura menghentikan pipeline (error), bukan auto-imputasi.


In [ ]:
print('missing per kolom kunci:')
display(destinations[['price', 'rating', 'time_minutes', 'lat', 'long', 'category', 'description']].isna().sum().rename('n_missing'))
valid = destinations.loc[destinations['rating'].between(1, 5), 'rating']
median_rating = float(valid.median())
print('median rating teramati:', median_rating, 'dari', len(valid), 'baris')
destinations['c2_rating'] = destinations['rating'].astype(float)
destinations['c2_rating_for_model'] = destinations['c2_rating'].where(destinations['c2_rating'].between(1, 5))
destinations['rating_imputed'] = (destinations['place_id'].eq(438)
    & destinations['place_name'].eq('Taman Hutan Raya Banten') & destinations['rating'].isna())
assert np.isfinite(median_rating)
destinations.loc[destinations['rating_imputed'], 'c2_rating_for_model'] = median_rating
destinations['rating_model_note'] = ''
destinations.loc[destinations['rating_imputed'], 'rating_model_note'] = (
    'Imputasi median rating teramati; bukan rating destinasi yang terverifikasi.')
imputation = {'method': 'median_observed_valid_ratings', 'value': median_rating,
    'observed_rows': len(valid),
    'imputed_ids': destinations.loc[destinations['rating_imputed'], 'place_id'].astype(int).tolist(),
    'calculation_column': 'c2_rating_for_model', 'source_rating_unchanged': True}
print('imputasi:', imputation)
assert destinations['c2_rating_for_model'].between(1, 5).all()
print('OK: 443 rating model valid 1-5; rating sumber Tahura tetap NaN')


## 7. Dedup dan agregat interaksi pengguna
Algoritma: snake_case header, coerce numerik, buang baris tanpa user/place atau rating di luar 1-5, drop_duplicates user x place keep-last, agregat mean/count/std per place_id (std NaN jadi 0), left-merge ke destinasi tanpa mengubah 443 baris.


In [ ]:
ratings = interaksi_raw.copy()
ratings.columns = [snake_case(c) for c in ratings.columns]
for c in ['user_id', 'place_id', 'place_ratings']:
    ratings[c] = pd.to_numeric(ratings[c], errors='coerce')
n_bad = int((ratings['user_id'].isna() | ratings['place_id'].isna() | ~ratings['place_ratings'].between(1, 5)).sum())
print('baris invalid interaksi dibuang (bukan destinasi):', n_bad)
ratings = ratings.loc[ratings['user_id'].notna() & ratings['place_id'].notna() & ratings['place_ratings'].between(1, 5)]
n_dup = int(ratings.duplicated(subset=['user_id', 'place_id']).sum())
ratings = ratings.drop_duplicates(['user_id', 'place_id'], keep='last')
print('dedup user x place keep-last:', n_dup, '| sisa:', len(ratings))
agg = ratings.groupby('place_id', as_index=False).agg(user_rating_mean=('place_ratings', 'mean'), user_rating_count=('place_ratings', 'count'), user_rating_std=('place_ratings', 'std'))
agg['user_rating_std'] = agg['user_rating_std'].fillna(0)
print('mencakup destinasi:', agg['place_id'].nunique(), 'dari 443')
display(agg.head())
destinations = destinations.merge(agg, on='place_id', how='left', validate='one_to_one')
check_ids(destinations, 'post-merge')
print('merge OK: tetap 443 baris')


## 8. Standardisasi wilayah dan kategori
Algoritma: city ke province via kamus 9 kota; category casefold ke category_clean via 6 mapping. Nilai tak terpetakan = error. Simpan category_original untuk audit.


In [ ]:
CITY_TO_PROVINCE = {'Jakarta': 'DKI Jakarta', 'Bandung': 'Jawa Barat', 'Semarang': 'Jawa Tengah',
    'Yogyakarta': 'DI Yogyakarta', 'Surabaya': 'Jawa Timur', 'Pandeglang': 'Banten',
    'Lebak': 'Banten', 'Purworejo': 'Jawa Tengah', 'Ngawi': 'Jawa Timur'}
CATEGORY_MAP = {'budaya': 'budaya', 'taman hiburan': 'hiburan', 'cagar alam': 'alam',
    'pusat perbelanjaan': 'belanja', 'tempat ibadah': 'religi', 'bahari': 'bahari'}
destinations['province'] = destinations['city'].map(CITY_TO_PROVINCE).astype('string')
destinations['category_original'] = destinations['category']
destinations['category_clean'] = destinations['category'].str.casefold().map(CATEGORY_MAP).astype('string')
print('kota tak terpetakan:', destinations.loc[destinations['province'].isna(), 'city'].unique().tolist())
print('kategori tak terpetakan:', destinations.loc[destinations['category_clean'].isna(), 'category'].unique().tolist())
assert destinations['province'].notna().all()
assert destinations['category_clean'].notna().all()
print('provinsi:', sorted(destinations['province'].unique().tolist()))
print('kategori:', sorted(destinations['category_clean'].unique().tolist()))
display(destinations[['place_id', 'place_name', 'city', 'province', 'category_original', 'category_clean']].tail(8))


## 9. Validasi geografis
Algoritma: pastikan lat -90 sampai 90 dan long -180 sampai 180. Tandai konflik label vs koordinat (Marina ID9). Koordinat tidak diubah; cukup flag dan catatan untuk review jarak/biaya dan peta.


In [ ]:
print('lat range:', destinations['lat'].min(), '-', destinations['lat'].max())
print('long range:', destinations['long'].min(), '-', destinations['long'].max())
flag = destinations.loc[destinations['coordinate_review_required']]
print('koordinat perlu review:', len(flag))
display(flag[['place_id', 'place_name', 'province', 'lat', 'long', 'coordinate_review_note']])
assert destinations['lat'].between(-90, 90).all()
assert destinations['long'].between(-180, 180).all()
print('OK: semua koordinat dalam range; Marina dipertahankan + flag')


## 10. Outlier harga tiket — cap P99 hanya untuk fitur
Algoritma: hitung persentil-99 harga master. Kolom c1_ticket_price = price (master utuh). Versi capped min(price, P99) hanya masuk Z-score. Harga 0 = gratis valid. Negatif/NaN = error.


In [ ]:
assert (destinations['price'] >= 0).all() and destinations['price'].notna().all()
cap_p99 = float(np.percentile(destinations['price'].to_numpy(dtype=float), 99))
print('P99 harga: Rp', format(cap_p99, ',.0f'))
print('harga > P99:', int((destinations['price'] > cap_p99).sum()))
print('harga max master (tetap): Rp', format(destinations['price'].max(), ',.0f'))
destinations['c1_ticket_price'] = destinations['price'].astype(float)
display(destinations[['place_id', 'place_name', 'c1_ticket_price']].sort_values('c1_ticket_price', ascending=False).head())


## 11. Fitur fasilitas C4 + activity tags
Algoritma C4: 6 kelompok (toilet, parkir, makan, ibadah, aksesibilitas, informasi) dideteksi via pencocokan kata utuh (word-boundary) pada deskripsi ternormalisasi. Lalu 16 aturan koreksi konteks (ID + nama + frasa cocok serentak) menghapus false-positive: negasi, rencana, tetangga, perbandingan arsitektur, nama batu, keluhan parkir. Skor c4 = rata-rata 6 flag. Tags: 12 pola hobi + warisan kategori. 0 = tidak ada bukti deskripsi, BUKAN tidak ada fasilitas.


In [ ]:
FACILITY_KEYWORDS = {
    'toilet': ['toilet', 'kamar mandi', 'wc umum'],
    'parking': ['parkir', 'parkiran', 'parking'],
    'food': ['warung', 'restoran', 'rumah makan', 'kafe', 'cafe', 'kuliner'],
    'worship': ['mushola', 'musala', 'masjid', 'tempat ibadah', 'gereja', 'pura', 'vihara'],
    'accessibility': ['disabilitas', 'difabel', 'kursi roda', 'wheelchair', 'aksesibel'],
    'information_center': ['pusat informasi', 'information center', 'layanan informasi'], }
ACTIVITY_KEYWORDS = {
    'hiking': ['hiking', 'mendaki', 'pendakian', 'trekking'],
    'fotografi': ['fotografi', 'spot foto', 'berfoto', 'pemandangan', 'panorama'],
    'snorkeling': ['snorkeling', 'snorkel'], 'diving': ['diving', 'menyelam', 'selam'],
    'camping': ['camping', 'berkemah', 'bumi perkemahan'],
    'kuliner': ['kuliner', 'makanan khas', 'jajanan', 'warung', 'restoran'],
    'sejarah': ['sejarah', 'bersejarah', 'peninggalan', 'museum', 'monumen'],
    'budaya': ['budaya', 'tradisi', 'kesenian', 'keraton'],
    'belanja': ['belanja', 'pusat perbelanjaan', 'pasar', 'mal', 'mall'],
    'berenang': ['berenang', 'kolam renang', 'waterpark', 'water park'],
    'edukasi': ['edukasi', 'pendidikan', 'belajar', 'museum'],
    'religi': ['ziarah', 'religi', 'ibadah', 'masjid', 'gereja', 'pura', 'vihara'],
    'rekreasi_keluarga': ['keluarga', 'wahana', 'taman bermain', 'taman hiburan'], }
CATEGORY_TAGS = {'budaya': {'budaya'}, 'belanja': {'belanja'}, 'religi': {'religi'}, 'hiburan': {'rekreasi_keluarga'}}
desc = destinations['description'].map(normalize_text)
fac_cols = []
for fac, keys in FACILITY_KEYWORDS.items():
    c = 'facility_' + fac + '_mentioned'
    destinations[c] = desc.map(lambda t, ks=keys: int(any(contains_keyword(t, k) for k in ks)))
    fac_cols.append(c)
destinations['facility_keyword_score'] = destinations[fac_cols].mean(axis=1)
destinations['facility_description_missing'] = desc.eq('').astype(int)
CTX = {
    18: ('Museum Bank Indonesia', 'gereja ini dibongkar', ['worship'], 'Gereja dalam cerita sejarah telah dibongkar.'),
    57: ('Taman Lapangan Banteng', 'tidak jauh dari gereja katedral', ['worship'], 'Gereja adalah lokasi lain di dekat taman.'),
    94: ('Sumur Gumuling', 'pada masa lalu', ['worship'], 'Fungsi ibadah masa lalu, bukan fasilitas aktif terkonfirmasi.'),
    115: ('Monumen Sanapati', 'di taman depan gereja', ['worship'], 'Gereja adalah penanda lokasi di luar monumen.'),
    149: ('Goa Cerme', 'rencana pendirian masjid', ['worship'], 'Masjid disebut dalam cerita sejarah.'),
    176: ('Museum Gunung Merapi', 'ke depan juga akan dilengkapi', ['parking', 'information_center'], 'Parkir disebut sebagai rencana; layanan informasi adalah fungsi museum, bukan fasilitas terpisah yang dipastikan.'),
    213: ('Gedung Sate', 'gaya atap pura bali', ['worship'], 'Pura adalah perbandingan gaya arsitektur.'),
    232: ('Bukit Moko', 'kawasan bandung mampu menarik', ['food'], 'Kuliner dibahas sebagai daya tarik Bandung secara umum.'),
    249: ('Upside Down World Bandung', 'ruangan dengan tema', ['toilet'], 'Kamar mandi adalah ruangan bertema untuk foto.'),
    254: ('Teras Cikapundung BBWS', 'sebelumnya, taman ini', ['food'], 'Warung ada dalam cerita kondisi sebelumnya, bukan fasilitas saat ini yang dipastikan.'),
    270: ('Bukit Bintang', 'kuala lumpur, malaysia', ['food'], 'Deskripsi membahas tempat di negara lain; tidak dipakai sebagai bukti fasilitas record Bandung.'),
    295: ('Museum Nike Ardilla', 'hard rock cafe', ['food'], 'Cafe disebut sebagai perbandingan konsep desain.'),
    298: ('Gunung Lalakon', 'batu warung', ['food'], 'Batu Warung adalah nama batu, bukan warung makan.'),
    347: ('Taman Pandanaran', 'untuk mendapatkan parkir di sini cukup susah', ['parking'], 'Keluhan mencari parkir tidak mengonfirmasi area parkir destinasi.'),
    353: ('Taman Srigunting', 'di sisi barat terdapat gereja blenduk', ['worship'], 'Gereja adalah bangunan tetangga, bukan fasilitas taman.'),
    441: ('Pantai Jatimalang', 'bukan pintu masuk atau area parkir terverifikasi', ['parking'], 'Penyangkalan verifikasi koordinat, bukan bukti fasilitas parkir.'),
}
destinations['facility_review_note'] = ''
for pid, (nm, phr, rej, note) in CTX.items():
    hit = (destinations['place_id'].eq(pid) & destinations['place_name'].map(normalize_text).eq(normalize_text(nm)) & desc.str.contains(normalize_text(phr), regex=False))
    destinations.loc[hit, ['facility_' + k + '_mentioned' for k in rej]] = 0
    destinations.loc[hit, 'facility_review_note'] = note
destinations['c4_facility_score'] = destinations[fac_cols].mean(axis=1)
def tags(row):
    t = normalize_text(str(row['place_name']) + ' ' + str(row['description']))
    s = set(CATEGORY_TAGS.get(row['category_clean'], set()))
    for tag, keys in ACTIVITY_KEYWORDS.items():
        if any(contains_keyword(t, k) for k in keys):
            s.add(tag)
    return '|'.join(sorted(s))
destinations['activity_tags'] = destinations.apply(tags, axis=1)
destinations['feature_source_note'] = 'C4: indikator deskripsi dengan koreksi konteks tertentu; 0 bukan bukti fasilitas tidak tersedia. Activity_tags tetap heuristik.'
print('dengan bukti fasilitas:', int(destinations['c4_facility_score'].gt(0).sum()))
print('koreksi konteks:', int(destinations['facility_review_note'].ne('').sum()))
print('deskripsi kosong:', int(destinations['facility_description_missing'].sum()))
print('dengan activity_tags:', int(destinations['activity_tags'].ne('').sum()))
display(destinations.loc[destinations['facility_review_note'] != '', ['place_id', 'place_name', 'facility_review_note']].head(10))


## 12. Fitur model K-Means (Z-score + one-hot)
Algoritma: validasi kelengkapan, cap harga di P99, Z-score harga dan rating ((x-mean)/std, std 0 jadi 1), one-hot kategori abjad, place_id di depan sebagai kunci bukan fitur jarak. C4 dan metadata kualitas tidak masuk clustering.


In [ ]:
num = destinations[['c1_ticket_price', 'c2_rating_for_model']].to_numpy(dtype=float, copy=True)
assert np.isfinite(num).all() and (num[:, 0] >= 0).all()
assert ((num[:, 1] >= 1) & (num[:, 1] <= 5)).all()
assert destinations['category_clean'].notna().all()
num[:, 0] = np.minimum(num[:, 0], cap_p99)
mean, scale = num.mean(axis=0), num.std(axis=0)
scale[scale == 0] = 1
cats = sorted(destinations['category_clean'].unique().tolist())
feat_cols = ['c1_ticket_price_z', 'c2_rating_z'] + ['category_' + c for c in cats]
onehot = np.array([[int(v == c) for c in cats] for v in destinations['category_clean']])
kmeans_ready = pd.DataFrame(np.column_stack(((num - mean) / scale, onehot)), columns=feat_cols)
kmeans_ready.insert(0, 'place_id', destinations['place_id'].to_numpy())
params = {'feature_columns': feat_cols, 'categories': cats, 'ticket_price_cap_p99': cap_p99,
    'mean': mean.tolist(), 'scale': scale.tolist(),
    'excluded': 'C4 hanya SPK; indikator deskripsi bukan fasilitas lapangan terverifikasi.'}
print('kolom fitur:', feat_cols)
print('mean:', np.round(mean, 4).tolist(), '| scale:', np.round(scale, 4).tolist())
display(kmeans_ready.head())
display(kmeans_ready[feat_cols].describe().round(3))


## 13. Validasi akhir + ekspor + fingerprint
Algoritma: check_ids 443 untuk destinasi dan fitur, pastikan fitur finite, tulis 2 CSV (UTF-8-SIG, LF) plus summary dan manifest JSON berisi hash sumber/output, parameter scaler/P99/imputasi, dan fingerprint SHA-256.


In [ ]:
destinations['model_features_complete'] = True
# --- Samakan urutan kolom dengan pipeline kanonis agar byte-identik ---
COLUMN_ORDER = ['place_id', 'place_name', 'description', 'category', 'city', 'price',
    'rating', 'time_minutes', 'coordinate', 'lat', 'long', 'source_dataset', 'source_url',
    'rating_status', 'rating_source_url', 'rating_access_date', 'coordinate_review_required',
    'coordinate_review_note', 'data_quality_issues', 'province', 'category_original',
    'category_clean', 'user_rating_mean', 'user_rating_count', 'user_rating_std',
    'c1_ticket_price', 'c2_rating', 'c2_rating_for_model', 'rating_imputed', 'rating_model_note',
    'facility_toilet_mentioned', 'facility_parking_mentioned', 'facility_food_mentioned',
    'facility_worship_mentioned', 'facility_accessibility_mentioned',
    'facility_information_center_mentioned', 'facility_keyword_score',
    'facility_description_missing', 'facility_review_note', 'c4_facility_score',
    'activity_tags', 'feature_source_note', 'model_features_complete']
missing = [c for c in COLUMN_ORDER if c not in destinations.columns]
extra = [c for c in destinations.columns if c not in COLUMN_ORDER]
assert not missing and not extra, 'kolom menyimpang: missing=' + str(missing) + ' extra=' + str(extra)
destinations = destinations[COLUMN_ORDER]
check_ids(destinations, 'destinations final')
check_ids(kmeans_ready, 'features final')
assert np.isfinite(kmeans_ready[feat_cols].to_numpy(dtype=float)).all()
paths = {k: PROJECT_ROOT / v for k, v in OUTPUTS.items()}
for p in paths.values():
    p.parent.mkdir(parents=True, exist_ok=True)
destinations.to_csv(paths['destinations'], index=False, encoding='utf-8-sig', lineterminator='\n')
kmeans_ready.to_csv(paths['features'], index=False, encoding='utf-8-sig', lineterminator='\n')
payload = {'pipeline_version': PIPELINE_VERSION,
    'source_hashes': {rel: sha256_file(PROJECT_ROOT / rel) for rel in SOURCES.values()},
    'place_ids': destinations['place_id'].astype(int).tolist(),
    'feature_parameters': params, 'rating_imputation': imputation}
manifest = dict(payload)
manifest['pipeline_fingerprint'] = fingerprint(payload)
summary = {'destination_rows': 443, 'source_rows': {'kaggle': 437, 'additional_java': 6},
    'rows_lost': 0, 'missing_destination_ids': [],
    'missing_rating_ids': destinations.loc[destinations['rating'].isna(), 'place_id'].astype(int).tolist(),
    'imputed_rating_ids': imputation['imputed_ids'], 'rating_imputation': imputation,
    'coordinate_review_ids': destinations.loc[destinations['coordinate_review_required'], 'place_id'].astype(int).tolist(),
    'data_quality_review_ids': destinations.loc[destinations['data_quality_issues'].ne(''), 'place_id'].astype(int).tolist(),
    'model_features_complete_rows': 443, 'kmeans_training_ready': True,
    'rating_rows_valid': len(ratings),
    'cities': sorted(destinations['city'].unique().tolist()), 'categories': cats,
    'destinations_with_facility_evidence': int(destinations['c4_facility_score'].gt(0).sum()),
    'destinations_with_facility_keyword_matches': int(destinations['facility_keyword_score'].gt(0).sum()),
    'destinations_with_facility_context_review': int(destinations['facility_review_note'].ne('').sum()),
    'destinations_without_facility_description': int(destinations['facility_description_missing'].sum()),
    'destinations_with_activity_tags': int(destinations['activity_tags'].ne('').sum()),
    'facility_evidence_definition': 'Penyebutan deskripsi yang diterima; bukan fasilitas terverifikasi di lapangan.',
    'feature_parameters': params, 'pipeline_fingerprint': manifest['pipeline_fingerprint'],
    'missing_value_policy': 'Pertahankan 443 destinasi; imputasi hanya rating model Tahura. Time_Minutes boleh kosong.'}
paths['summary'].write_text(json.dumps(summary, indent=2, ensure_ascii=False, allow_nan=False) + '\n', encoding='utf-8', newline='\n')
manifest_full = dict(manifest)
manifest_full['output_hashes'] = {OUTPUTS['destinations']: sha256_file(paths['destinations']), OUTPUTS['features']: sha256_file(paths['features']), OUTPUTS['summary']: sha256_file(paths['summary'])}
paths['manifest'].write_text(json.dumps(manifest_full, indent=2, ensure_ascii=False, allow_nan=False) + '\n', encoding='utf-8', newline='\n')
for k, p in paths.items():
    print(k, ':', p)
print('ID tersimpan lengkap: 1-443')
print('Fingerprint:', manifest['pipeline_fingerprint'])


## Catatan reproduksibilitas
- Hash sumber/output + fingerprint di manifest menjamin preprocessing, impor DB, dan training memakai artifact yang sama. Bila sumber berubah, jalankan ulang notebook ini lalu impor dan training ulang.
- kmeans_training_ready = True artinya fitur lengkap, BUKAN model sudah dilatih.
- Tidak menimpa data/raw, bukti data/review, maupun laporan DOCX.
- Batasan jujur: C4/tags heuristik deskripsi; rating tambahan = agregat Google Maps; Marina ID9 butuh review jarak; harga = snapshot dataset + Perda, bukan harga transaksi real-time.
